In [ ]:
!pip install tensorflow

In [11]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print(tf.__version__)

2.21.0


In [10]:
data = """One of my favorite things is travelling to places where time seems to move a little slower. I love the feeling of walking through unfamiliar streets, discovering small cafés, listening to conversations in languages I barely understand, and noticing the little details that make every place unique. Sometimes, the best part of a trip isn't visiting a famous monument but watching the sunset from a quiet corner or getting lost and stumbling upon something unexpected. I think travelling teaches us to appreciate different perspectives and reminds us how much there is to experience beyond our everyday routines. If I could, I would spend my life collecting little moments and stories from different corners of the world."""

lines = data.lower().split("\n")
print("Total lines:", len(lines))
print("Total words:", len(data.split()))

Total lines: 1
Total words: 116


In [12]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)

total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)
print("Sample word index:", dict(list(tokenizer.word_index.items())[:10]))

Vocabulary size: 88
Sample word index: {'to': 1, 'i': 2, 'the': 3, 'of': 4, 'a': 5, 'and': 6, 'little': 7, 'my': 8, 'is': 9, 'travelling': 10}


In [13]:
input_sequences = []

for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Total sequences:", len(input_sequences))
print("X shape:", X.shape, "| y shape:", y.shape)

Total sequences: 115
X shape: (115, 115) | y shape: (115,)


In [14]:
model = Sequential([
    Embedding(total_words, 64),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.build(input_shape=(None, max_len - 1))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 115, 64)        │         5,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │        66,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 88)             │         8,888 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,520 (314.53 KB)

 Trainable params: 80,520 (314.53 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(X, y, epochs=200, batch_size=16, verbose=0)

print("Final loss    :", round(history.history['loss'][-1], 4))
print("Final accuracy:", round(history.history['accuracy'][-1], 4))

Final loss    : 0.1343
Final accuracy: 1.0


In [16]:
index_word = {idx: word for word, idx in tokenizer.word_index.items()}

def sample_with_temperature(probs, temperature=0.7):
    probs = np.asarray(probs).astype('float64')
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs) / np.sum(np.exp(probs))
    return np.random.choice(len(probs), p=probs)

def generate_text(seed_text, next_words=8, temperature=0.7):
    result = seed_text.lower()
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([result])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')
        probs = model.predict(token_list, verbose=0)[0]
        next_index = sample_with_temperature(probs, temperature)
        next_word = index_word.get(next_index, "")
        if next_word == "":
            break
        result += " " + next_word
    return result

seeds = ["cricket", "every", "young"]

print("Generated lines:\n")
for seed in seeds:
    print("-", generate_text(seed, next_words=8))

Generated lines:

- cricket of my favorite things is travelling travelling to
- every of my favorite my travelling travelling to places
- young of my favorite things is travelling to places


Step	What happens
Tokenizer	Converts each word to an integer to build a vocabulary
Input sequences	Each line is split into growing prefixes so the model learns to predict the next word
Padding	Makes all sequences the same length
Embedding + LSTM	Converts words to vectors, then learns word order and context
Dense (softmax)	Gives a probability for every word in the vocabulary
Generation	Predicts the next word, appends it, and repeats

In [17]:
import os
print(os.listdir())

['.config', 'sample_data']
